# 🎙️ Fine-Tune Piper TTS Tiếng Việt - Giọng Đọc Cực Nhanh (Offline 100%)

Notebook này hỗ trợ bạn huấn luyện thêm (Fine-tune) mô hình **Piper VITS Tiếng Việt** dựa trên Checkpoint gốc đã học sẵn hơn 4,700 epochs của cộng đồng Rhasspy.

### Ưu điểm:
- **Thời gian cực nhanh:** Chỉ cần 100-300 câu mẫu (30-60 phút thu âm).
- **Fine-tune trên T4 GPU (Google Colab):** Mất khoảng 1 - 2 tiếng.
- **Xuất trực tiếp sang `.onnx`:** Tốc độ suy luận chỉ **50 - 90ms** trên máy tính cá nhân, hoàn toàn không cần Internet!

## 🛠️ Bước 1: Kiểm tra GPU & Cài đặt môi trường

In [ ]:
# Kiểm tra GPU được cấp phát (Nên chọn T4 GPU)
!nvidia-smi

# Cài đặt các gói hệ thống cần thiết (espeak-ng cho phonemizer tiếng Việt)
!apt-get update -qq
!apt-get install -y -qq espeak-ng ffmpeg

# Clone mã nguồn Piper Training
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git
%cd piper/src/python

# Cài đặt thư viện huấn luyện
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -r requirements.txt
!pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 piper-phonemize onnxruntime

## 📥 Bước 2: Tải Checkpoint Tiếng Việt gốc (Base Model)
Chúng ta kế thừa từ checkpoint `vi_VN-vais1000-medium` đã được huấn luyện sẵn 4,769 epochs.

In [ ]:
%cd /content
!mkdir -p /content/base_model

# Tải checkpoint gốc và file config
!wget -q --show-progress -c "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/vi/vi_VN/vais1000/medium/epoch%3D4769-step%3D919580.ckpt" -O /content/base_model/base.ckpt
!wget -q --show-progress -c "https://huggingface.co/datasets/rhasspy/piper-checkpoints/raw/main/vi/vi_VN/vais1000/medium/config.json" -O /content/base_model/config.json

print("✅ Đã tải xong Base Checkpoint tiếng Việt!")

## 📤 Bước 3: Nạp Tập Dữ Liệu Âm Thanh Của Bạn (Dataset)
Chuẩn bị một file nén `dataset.zip` có cấu trúc:
```
dataset/
  ├── wavs/
  │    ├── 0001.wav
  │    ├── 0002.wav
  │    └── ...
  └── metadata.csv (định dạng: 0001.wav|Câu đọc tiếng Việt)
```

In [ ]:
from google.colab import files
import os

print("👉 Bấm nút bên dưới để chọn tải lên tệp 'dataset.zip':")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.zip'):
        !unzip -q -o "{fn}" -d /content/my_dataset
        print(f"✅ Đã giải nén {fn} thành công!")
        break

## ⚙️ Bước 4: Tiền xử lý dữ liệu (Phonemize & Feature Extraction)

In [ ]:
%cd /content/piper/src/python
!python3 -m piper_train.preprocess \
    --language vi \
    --input-dir /content/my_dataset \
    --output-dir /content/training_data \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050

# Sao chép cấu hình phoneme từ base model
!cp /content/base_model/config.json /content/training_data/config.json
print("✅ Tiền xử lý hoàn tất! Sẵn sàng huấn luyện.")

## 🚀 Bước 5: Chạy Fine-Tuning (Huấn luyện thêm)

In [ ]:
%cd /content/piper/src/python

# Chạy fine-tune 1000 epochs (mất khoảng 1 - 2 tiếng trên Colab)
!python3 -m piper_train \
    --dataset-dir /content/training_data \
    --accelerator 'gpu' \
    --devices 1 \
    --batch-size 8 \
    --validation-split 0.05 \
    --num-test-examples 2 \
    --max_epochs 1000 \
    --resume_from_checkpoint /content/base_model/base.ckpt \
    --checkpoint-epochs 25 \
    --precision 32

## 📦 Bước 6: Xuất Mô Hình Sang Định Dạng ONNX
Định dạng `.onnx` là định dạng siêu nhẹ giúp Alexa trên Arch Linux đọc chữ với độ trễ <100ms.

In [ ]:
import glob

# Tìm checkpoint mới nhất vừa train xong
checkpoints = glob.glob("/content/piper/src/python/lightning_logs/**/checkpoints/*.ckpt", recursive=True)
if not checkpoints:
    print("❌ Không tìm thấy checkpoint sau khi train.")
else:
    latest_ckpt = max(checkpoints, key=os.path.getctime)
    print(f"🎯 Đang xuất từ checkpoint: {latest_ckpt}")
    
    !python3 -m piper_train.export_onnx \
        "{latest_ckpt}" \
        /content/vi_VN-custom_voice-medium.onnx
    
    # Copy config.json đi kèm
    !cp /content/training_data/config.json /content/vi_VN-custom_voice-medium.onnx.json
    
    print("🎉 XUẤT THÀNH CÔNG!")
    print("File 1: /content/vi_VN-custom_voice-medium.onnx")
    print("File 2: /content/vi_VN-custom_voice-medium.onnx.json")

## 💾 Bước 7: Tải mô hình về máy tính

In [ ]:
from google.colab import files

files.download('/content/vi_VN-custom_voice-medium.onnx')
files.download('/content/vi_VN-custom_voice-medium.onnx.json')